In [ ]:
from qiskit import QuantumCircuit, transpile, ClassicalRegister
from qiskit.quantum_info import Statevector, DensityMatrix, partial_trace, entropy, state_fidelity, random_statevector
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit.providers.fake_provider import GenericBackendV2
from IPython.display import display
import matplotlib.pyplot as plt
import sympy
import numpy as np

In the following, I build a circuit of 8 qubits which I use to build the four orthogonal Bell states. Afterwords, I use a GenericBackend to translate the circuit into real quantum hardware, where virtual qubits map into physical qubits on the chip, and the transpiler optimizes the operations, and uses native single-qubit basis gates: phase rotations, $R_{z}$ and $\sqrt{X}$

In [ ]:
## All the four Bell states
qc = QuantumCircuit(8)
# Phi+
qc.h(0); qc.cx(0,1)
# Phi-
qc.h(2); qc.cx(2,3); qc.z(2)
# Psi+
qc.h(4); qc.cx(4,5); qc.x(4)
# Psi-
qc.h(6); qc.cx(6,7); qc.x(6); qc.z(6)

# On a real hardware 
backend = GenericBackendV2(num_qubits=20)

# Transpile the 8-qubit ideal circuit for that hardware
transpiled_qc = transpile(qc, backend=backend, optimization_level=3)

# Drawing the two circuits side by side
fig, axs = plt.subplots(1, 2, figsize=(16, 8))

qc.draw(output='mpl', style='iqp', ax=axs[0])
axs[0].set_title("Ideal Circuit (Virtual Qubits)", fontsize=16)

transpiled_qc.draw(output='mpl', style='iqp', ax=axs[1])
axs[1].set_title("Hardware Reality (Physical Basis Gates & Routing)", fontsize=16)

plt.tight_layout()
plt.show()

I can also add measurement gates. Reading out a qubit requires a microwave pulse sent down a readout resonator, which takes significantly longer than single-qubit logic gates.

In [ ]:
# Append measurements to the ideal 8-qubit circuit
qc_measured = qc.copy()
qc_measured.measure_all()  # Adds an 8-bit classical register and measures everything

# Re-transpile using the same backend target profile
backend = GenericBackendV2(num_qubits=20)
transpiled_measured_qc = transpile(qc_measured, backend=backend, optimization_level=3)

transpiled_measured_qc.draw(output='mpl', style='iqp')


And a quick simulation on AerSimulator. The result is an evenly distributed mix of binary strings. Note that Qiskit orders bitstrings from highest qubit to lowest qubit (q7 q6 q5 q4 q3 q2 q1 q0), the 8-bit output breaks down into four independent 2-bit Bell state pairs:
$$ \Psi^- \equiv q7 q6\\ 
   \Psi^+ \equiv q5 q4\\
    \Phi^- \equiv q3 q2\\
   \Phi^+ \equiv q1 q0\\$$

Note that a measurement will collapse each Bell state to one of the two bits in the superposition (eg $\ket{00}$ or $\ket{11}$ for $\Phi^{\pm}$). Thus we expect mixtures of bit-strings that look like 01100011, thus all 16 strings have roughly the same counts. 
 

In [ ]:
# Simulation
simulator = AerSimulator()
result = simulator.run(transpiled_measured_qc, shots=2048).result()
counts = result.get_counts()

# Qiskit layout order: q7q6 q5q4 q3q2 q1q0
grouped_counts = {
    f"{k[0:2]} {k[2:4]} {k[4:6]} {k[6:8]}": v 
    for k, v in counts.items()
}

# Histogram
fig, ax = plt.subplots(figsize=(12, 6))
plot_histogram(grouped_counts, ax=ax, title="Parallel Bell States Measurement Outcomes")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


One may use decoding where we transform the qubit into an actual 2-bit string. In this case, the histogram would collapse into a single measurement. This is the idea also behind superdense coding, which we will demonstrate right after. Essentially, one encodes a message (two classical bits) into a qubit (a Bell state). After decoding, the histogram spikes at exactly where the original message was. The decoder is for each pair a CNOT gate followed by a Hadamard gate on the top qubit.

Note that the result
$$ 11100100 $$
means that after decoding, we get all the original Bell states back!

In [ ]:
# Initially we have the 8-qubit circuit with the four Bell states
qc_decoded = qc.copy()

qc_decoded.barrier() # Visual separator for the decoder section

# The decoders enter here per pair
# Phi+
qc_decoded.cx(0,1) 
qc_decoded.h(0)
# Phi-
qc_decoded.cx(2,3) 
qc_decoded.h(2)
# Psi+
qc_decoded.cx(4,5) 
qc_decoded.h(4)
# Psi-
qc_decoded.cx(6,7) 
qc_decoded.h(6)

qc_decoded.measure_all()

# Transpilation and simulation
simulator = AerSimulator()
transpiled_measured_qc = transpile(qc_decoded, simulator)
counts = simulator.run(transpiled_measured_qc, shots=1024).result().get_counts()

# Reformat keys for readability (q7q6 q5q4 q3q2 q1q0)
grouped_counts = {f"{k[0:2]} {k[2:4]} {k[4:6]} {k[6:8]}": v for k, v in counts.items()}

fig, ax = plt.subplots(figsize=(8, 5))
plot_histogram(grouped_counts, ax=ax, title="Decoded Bell States (Deterministic Outcomes)")
plt.tight_layout()
plt.show()


# Superdense Coding
This protocol allows for the transmission of two classical bits using one qubit of quantum communication (at the cost of one e-bit of entanglement). The main idea is simply that Alice effectively chooses which Bell state she would like to be sharing with Bob, she sends Bob her qubit, and Bob measures to determine which Bell state Alice chose. The protocol is as follows:

Given a **2-bit string** `bits`, the quantum circuit performs the following steps:
1. **Create a Bell pair**
   - Qubit `0` belongs to Alice.
   - Qubit `1` belongs to Bob.
2. **Alice encodes her 2-bit message** on **qubit `0` only**:

   | Message | Operation |
   |:-------:|:---------:|
   | `00` | Do nothing ($I$) |
   | `01` | Apply $X$ |
   | `10` | Apply $Z$ |
   | `11` | Apply $Z$, then $X$ |

3. **Alice sends her qubit to Bob**
   - Nothing needs to be implemented in the circuit.
   - Conceptually, Alice hands qubit `0` to Bob.

4. **Bob decodes the message**
   - Apply **CNOT** with `qubit 0` as the control and `qubit 1` as the target.
   - Apply the **Hadamard gate** $H$ to qubit `0`.

5. **Measure both qubits**
   - Bob measures qubits `0` and `1`.

In [ ]:
def superdense_coding(bits):
    qc = QuantumCircuit(2, 2)
    # Create Bell pair
    qc.h(0)
    qc.cx(0,1)
    # Alice encodes her message on her qubit
    match bits:
        case "00":
            pass
        case "01":
            qc.x(0)
        case "10":
            qc.z(0)
        case "11":
            qc.z(0)
            qc.x(0)
        case _:
            raise ValueError("Invalid bit string")
    sv = Statevector.from_instruction(qc)
    print("Bell State after encoding")
    display(sv.draw('latex'))

    # Bob does CNOT
    qc.cx(0,1)
    qc.h(0)

    # Measure
    qc.measure([0, 1], [0, 1])  # measure qubit 0->cbit 0, qubit 1->cbit 1

    return qc

sim = AerSimulator()

for message in ['00', '01', '10', '11']:
    qc = superdense_coding(message)
    tqc = transpile(qc, sim)
    fig = tqc.draw(output="mpl")

    result = sim.run(tqc, shots=1024).result()
    counts = result.get_counts()
    print(f"Sent: {message}  ->  Measured: {counts}")
    fig.savefig(rf"circuit_{message}.png")


# Quantum Teleportation
The scenario: Alice has an unknown qubit  (called qubit 0) $$ ∣ψ⟩=α∣0⟩+β∣1⟩ $$
Alice and Bob pre-share a Bell pair (qubit 1 = Alice's half, qubit 2 = Bob's half). Alice wants Bob to end up holding $
\ket{\psi} $ using only a classical channel.

In [ ]:
def teleport(psi:Statevector) -> QuantumCircuit:
    qc = QuantumCircuit(3, 2)
    qc.initialize(psi, 0)
    qc.h(1)
    qc.cx(1, 2)
    # Alice Bell Basis measurement
    qc.cx(0, 1)  # control = unknown state, target = Alice's Bell qubit
    qc.h(0)
    qc.measure([0, 1], [0, 1])  # measure qubit 0->cbit 0, qubit 1->cbit 1
    # Alice shares classical bits
    # Bob corrections on qubit 2 based on Alice's classical bits:
    with qc.if_test((qc.clbits[1], 1)):
        qc.x(2)  # if cbit 1 == 1, apply X on Bob's qubit
    with qc.if_test((qc.clbits[0], 1)):
        qc.z(2)  # if cbit 0 == 1, apply Z on Bob's qubit

    qc.save_statevector()
    fig = qc.draw(output = 'mpl')
    display(fig)
    return qc

sv = random_statevector(2)
qc = teleport(sv)

simulator = AerSimulator(method="statevector")

result = simulator.run(qc).result()

final_state = result.get_statevector()
# Trace out Alice's qubits (0 and 1)
bob_state = partial_trace(final_state, [0, 1])

fidelity = state_fidelity(bob_state, sv)

print(f"Fidelity: {fidelity}")

## Wiesner's Quantum Money

Now let us look at a protocol known as Wiesner's Quantum Money. A banknote is given by a serial number and a quantum circuit $(s,qc)$. The serial number contains two classical strings (a,b) stored in a ledger, and only the bank knows. Each banknote has a random unique identifier written on it.

Note that a valid note on this noiseless simulator (with one shot) returns a dictionary with a single bitstring and a count of 1. The result is completely deterministic because:
- If the qubits are in the $Z$ basis ($a_i = 0$), the state is $\ket{0}$ or $\ket{1}$ and the measurement in the $Z$ basis will give $b_i$.
- If the qubits are in the $X$ basis ($a_i = 1$), the state is $\ket{+}$ or $\ket{-}$ and we first apply $H$ to rotate to the $Z$ basis and then measure to get $b_i$.

Now a forger has the banknote $(s, qc)$ but not the ledger so they don't know the string $a$. They will start measuring each qubit in a randomly chosen basis and then re-prepare the state based on what they measured.

In [ ]:
import uuid

ledger = {} # a dict to map a serial number into the strings (a,b)

def mint(n: int):
    # Returns a serial number with the ledger stored
    # alongside the quantum circuit
    a = np.random.randint(0, 2, n)
    b = np.random.randint(0, 2, n)

    qc = QuantumCircuit(n)
    for i in range(n):
        if a[i] == 0 and b[i] == 0:
            pass
        elif a[i] == 0 and b[i] == 1:
            qc.x(i)
        elif a[i] == 1 and b[i] == 0:
            qc.h(i)
        elif a[i] == 1 and b[i] == 1:
            qc.x(i); qc.h(i)

    # the random unique identifier
    s = str(uuid.uuid4()) 
    ledger[s] = (a,b)
    return qc, s

def verify(s, qc):
    a, b = ledger[s]
    n = len(a)
    qc.add_register(ClassicalRegister(n))

    for i in range(n):
        if a[i] == 1:
            qc.h(i)

    qc.measure(list(range(n)), list(range(n)))    

    # Simulation - note only one shot - which is what the Bank does!
    simulator = AerSimulator()
    result = simulator.run(qc, shots=1).result()
    counts = result.get_counts()

    bitstring = list(counts)[0][::-1] # correct for qiskit endianess

    return all(int(bitstring[i]) == int(b[i]) for i in range(len(b)))
    
def forge(qc, n):
    forged_qc = qc.copy()
    forged_qc.add_register(ClassicalRegister(n))

    bases = np.random.randint(0, 2, n) # 0 = Z, 1 = X
    
    for i in range(n):
        if bases[i] == 1:
            forged_qc.h(i)

    forged_qc.measure(list(range(n)), list(range(n)))    
    
    # Simulation - note only one shot - which is what the Bank does!
    simulator = AerSimulator()
    result = simulator.run(forged_qc, shots=1).result()
    counts = result.get_counts()

    bitstring = list(counts)[0][::-1] # correct for qiskit endianess

    forged_note = QuantumCircuit(n)
    for i in range(n):
        bit = int(bitstring[i])
        if bases[i] == 0 and bit == 0:
            pass           # |0>
        elif bases[i] == 0 and bit == 1:
            forged_note.x(i)   # |1>
        elif bases[i] == 1 and bit == 0:
            forged_note.h(i)   # |+>
        elif bases[i] == 1 and bit == 1:
            forged_note.x(i)
            forged_note.h(i)   # |->

    return forged_note

n = 10
qc, s = mint(n)

accepted = 0
for _ in range(100):
    forged_qc = forge(qc, n)
    #print(forged_qc)  # check it looks right
    #print(verify(s, forged_qc))
    #break  # just one iteration for now
    if verify(s, forged_qc):
        accepted += 1

print(f"Acceptance rate: {accepted/1000:.2f}")
print(f"Theoretical prediction (3/4)^n: {(3/4)**n:.4f}")